In [1]:
#30 FEATURE
# gerekli kütüphaneleri yükledim
import pandas as pd
import numpy as np
import warnings
import os
import re
import matplotlib.pyplot as plt
import openpyxl

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from mrmr import mrmr_classif

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    make_scorer, accuracy_score, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score
)

warnings.filterwarnings("ignore")


# ROC-AUC güvenli kısmı sorun çözücü şekilde
EPS = 1e-6

def _get_scores_from_estimator(estimator, X):
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        if np.ndim(s) > 1:
            s = s[:, 0]
        return s

    return estimator.predict(X)

def safe_roc_auc_estimator(estimator, X, y_true):
    try:
        y_score = _get_scores_from_estimator(estimator, X)
        auc = roc_auc_score(y_true, y_score)
        if np.isnan(auc):
            return 0.5
        if auc == 0.0:
            return EPS
        return float(auc)
    except Exception:
        return 0.5


# mRMR için olan eklentiler
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=30):
        self.K = K
        self.selected_features_ = None
        self.selected_indices_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy().reset_index(drop=True)
            self.feature_names_in_ = list(X_df.columns)
        else:
            X_df = pd.DataFrame(
                X,
                columns=[f"f{i}" for i in range(X.shape[1])]
            ).reset_index(drop=True)
            self.feature_names_in_ = list(X_df.columns)

        y_sr = pd.Series(y).reset_index(drop=True)

        if len(X_df) != len(y_sr):
            raise ValueError(
                f"X ve y örnek sayıları eşit değil: "
                f"X={len(X_df)}, y={len(y_sr)}"
            )

        self.selected_features_ = mrmr_classif(
            X=X_df,
            y=y_sr,
            K=self.K
        )

        self.selected_indices_ = [
            X_df.columns.get_loc(column)
            for column in self.selected_features_
        ]

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()
        else:
            X_df = pd.DataFrame(
                X,
                columns=self.feature_names_in_
            )

        return X_df[self.selected_features_]

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X[self.selected_features_]
        else:
            X_df = pd.DataFrame(X, columns=self.feature_names_in_)
            return X_df[self.selected_features_]
          


# TOPSIS Analiz için ek şeyler
def topsis_analysis(results_df, metric_columns, weights):
    """
    Verilen bir DataFrame'e TOPSIS analizi uygular ve sıralanmış sonuçları döndürür.
    Sıfıra bölme hatalarına karşı korumalıdır.
    """
    print("--- TOPSIS Analizi Başlatılıyor ---")

    decision_matrix = results_df[metric_columns].apply(pd.to_numeric, errors='coerce').fillna(0).values

    denominator = np.sqrt(np.sum(decision_matrix**2, axis=0)) + 1e-8
    norm_matrix = decision_matrix / denominator

    weighted_matrix = norm_matrix * np.array(weights)

    ideal_best = np.max(weighted_matrix, axis=0)
    ideal_worst = np.min(weighted_matrix, axis=0)

    dist_to_best = np.sqrt(np.sum((weighted_matrix - ideal_best)**2, axis=1))
    dist_to_worst = np.sqrt(np.sum((weighted_matrix - ideal_worst)**2, axis=1))

    topsis_score = dist_to_worst / (dist_to_best + dist_to_worst + 1e-8)

    ranked_df = results_df.copy()
    ranked_df['Topsis_Score'] = topsis_score
    ranked_df['Rank'] = ranked_df['Topsis_Score'].rank(ascending=False, method='min').astype(int)

    return ranked_df.sort_values(by='Rank', ascending=True).reset_index(drop=True)


# EKLENDİ:
# Seçilen miRNA'ların tümörde mi normalde mi daha yüksek olduğunu hesaplar
def get_biomarker_direction_for_selected_mirnas(X, y, selected_mirnas, tumor_label=1, normal_label=0):
    """
    Seçilen miRNA'ların tümörde mi normalde mi daha yüksek olduğunu hesaplar.

    tumor_label = 1  (TP)
    normal_label = 0 (NT)

    Veri log2(RPM+1) olduğu için:
    log2FC_Tumor_vs_Normal = mean(tumor) - mean(normal)
    """
    rows = []

    y = pd.Series(y).reset_index(drop=True)
    X = X.reset_index(drop=True)

    for mirna in selected_mirnas:
        if mirna not in X.columns:
            continue

        tumor_vals = X.loc[y == tumor_label, mirna]
        normal_vals = X.loc[y == normal_label, mirna]

        tumor_mean = tumor_vals.mean()
        normal_mean = normal_vals.mean()

        log2fc = tumor_mean - normal_mean

        if log2fc > 0:
            direction = "Up in Tumor"
            biomarker_type = "Positive biomarker candidate"
            interpretation = "Higher expression in tumor than normal"
        elif log2fc < 0:
            direction = "Down in Tumor"
            biomarker_type = "Negative biomarker candidate"
            interpretation = "Lower expression in tumor than normal"
        else:
            direction = "No Change"
            biomarker_type = "No clear biomarker direction"
            interpretation = "Similar expression in tumor and normal"

        rows.append({
            "miRNA": mirna,
            "Tumor_mean": tumor_mean,
            "Normal_mean": normal_mean,
            "log2FC_Tumor_vs_Normal": log2fc,
            "Direction": direction,
            "Biomarker_Type": biomarker_type,
            "Interpretation": interpretation
        })

    biomarker_df = pd.DataFrame(rows)
    if not biomarker_df.empty:
        biomarker_df = biomarker_df.sort_values("log2FC_Tumor_vs_Normal", ascending=False).reset_index(drop=True)
    return biomarker_df


# Dosya yolu ve parametreleri girdim
file_path = r"THCA_miRNA_unique_patients_only.csv"

param_grids = {
    "SVM": {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    "Log. Reg.": {
        'classifier__C': [0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2']
    },
    "Extra Trees": {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "LDA": [
        {
            'classifier__solver': ['svd']
        },
        {
            'classifier__solver': ['lsqr', 'eigen'],
            'classifier__shrinkage': [None, 'auto']
        }
    ],
    "Random Forest": {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "Gradient Boosting": {
        'classifier__n_estimators': [50, 100],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__max_depth': [2, 3]
    },
    "SGD": [
        {
            'classifier__alpha': [0.0001, 0.001, 0.01],
            'classifier__penalty': ['l1', 'l2']
        },
        {
            'classifier__alpha': [0.0001, 0.001, 0.01],
            'classifier__penalty': ['elasticnet'],
            'classifier__l1_ratio': [0.15, 0.5]
        }
    ],
    "KNN": {
        'classifier__n_neighbors': [3, 5, 7],
        'classifier__weights': ['uniform', 'distance'],
        'classifier__p': [1, 2]
    },
    "Decision Tree": {
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__max_depth': [None, 5, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "AdaBoost": {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
    }
}

# 30 tane feature seçtim 1881 miRNA'dan
K_FEATURES = 30
FEATURE_OUTPUT_EXCEL = "selected_features_by_scenario_30.xlsx"
PLOT_DIR = "selected_feature_plots_30"
RESULTS_EXCEL = "classification_results_leakage_fixed_30.xlsx"

TOPSIS_OUTPUT_EXCEL = "topsis_sonuclari_sirali_30.xlsx"
TOPSIS_SHEET_NAME = "Tum Sonuçlar (Sirali)"
METRICS_TO_EVALUATE = ['MCC', 'F1-Score', 'ROC AUC', 'Accuracy', 'Precision', 'Recall']
METRIC_WEIGHTS = [0.30, 0.30, 0.15, 0.10, 0.075, 0.075]


# smote undersampler ve unbalanced olayı burda
def make_safe_sheet_name(name, max_len=31):
    name = re.sub(r'[\\/*?:\[\]]', '_', name)
    return name[:max_len]

def scenario_code(data_handling, optimization_status, selector_name):
    handling_map = {
        "Unbalanced": "UB",
        "Downsampled": "DS",
        "Oversampled (SMOTE)": "OS"
    }
    opt_map = {
        "Unoptimized": "U",
        "Optimized": "O"
    }
    return f"{handling_map.get(data_handling, 'X')}_{opt_map.get(optimization_status, 'X')}_{selector_name}"

def get_selector(selector_name, k=30):
    if selector_name == "ANOVA":
        return SelectKBest(score_func=f_classif, k=k)
    elif selector_name == "MI":
        return SelectKBest(score_func=mutual_info_classif, k=k)
    elif selector_name == "mRMR":
        return MRMRSelector(K=k)
    else:
        raise ValueError(f"Bilinmeyen selector: {selector_name}")

def preprocess_for_selector(
    X,
    y,
    data_handling,
    random_state=42
):
    """
    Pipeline'ın selector öncesi kısmını taklit eder:
    scaler -> opsiyonel downsampling veya SMOTE.

    Bu fonksiyon feature incelemesinde yalnızca ilgili
    training fold üzerinde çağrılır.
    """

    X = X.copy().reset_index(drop=True)
    y_out = pd.Series(y).reset_index(drop=True)

    if len(X) != len(y_out):
        raise ValueError(
            f"X ve y örnek sayıları eşit değil: "
            f"X={len(X)}, y={len(y_out)}"
        )

    scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns
    ).reset_index(drop=True)

    if data_handling == "Downsampled":
        undersampler = RandomUnderSampler(
            random_state=random_state
        )

        X_res, y_res = undersampler.fit_resample(
            X_scaled,
            y_out
        )

        X_res = pd.DataFrame(
            X_res,
            columns=X.columns
        ).reset_index(drop=True)

        y_res = pd.Series(
            y_res
        ).reset_index(drop=True)

        return X_res, y_res

    if data_handling == "Oversampled (SMOTE)":
        smote = SMOTE(
            random_state=random_state
        )

        X_res, y_res = smote.fit_resample(
            X_scaled,
            y_out
        )

        X_res = pd.DataFrame(
            X_res,
            columns=X.columns
        ).reset_index(drop=True)

        y_res = pd.Series(
            y_res
        ).reset_index(drop=True)

        return X_res, y_res

    return X_scaled, y_out

def get_selected_features_from_selector(selector, X_columns):
    if isinstance(selector, MRMRSelector):
        selected = list(selector.selected_features_)
        scores_df = pd.DataFrame({
            "miRNA": selected,
            "score_or_rank": list(range(1, len(selected) + 1))
        })
        return selected, scores_df

    support = selector.get_support()
    selected = list(pd.Index(X_columns)[support])

    if hasattr(selector, "scores_") and selector.scores_ is not None:
        scores = selector.scores_
        score_map = pd.DataFrame({
            "miRNA": X_columns,
            "raw_score": scores
        })
        score_map = score_map[score_map["miRNA"].isin(selected)].copy()
        score_map["score_or_rank"] = score_map["raw_score"]
        score_map = score_map.sort_values("score_or_rank", ascending=False).reset_index(drop=True)
        return list(score_map["miRNA"]), score_map[["miRNA", "score_or_rank"]]

    scores_df = pd.DataFrame({
        "miRNA": selected,
        "score_or_rank": list(range(1, len(selected) + 1))
    })
    return selected, scores_df

def inspect_selected_features_for_scenario(
    X, y, groups, data_handling, selector_name, optimization_status, k=30, cv_splits=5, random_state=42
):
    """
    1) Tüm veri üzerinde seçilen feature'lar
    2) CV fold'larında seçilme sıklıkları
    """
    X_proc, y_proc = preprocess_for_selector(X, y, data_handling, random_state=random_state)
    selector_full = get_selector(selector_name, k=k)
    selector_full.fit(X_proc, y_proc)
    selected_full, selected_scores_df = get_selected_features_from_selector(selector_full, X_proc.columns)

    full_selected_df = selected_scores_df.copy()
    full_selected_df.insert(0, "data_handling", data_handling)
    full_selected_df.insert(1, "optimization", optimization_status)
    full_selected_df.insert(2, "selector", selector_name)

    groups = pd.Series(groups).reset_index(drop=True)

    cv = StratifiedGroupKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    freq_counter = pd.Series(0, index=X.columns, dtype=int)

    fold_details = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()

        X_train_proc, y_train_proc = preprocess_for_selector(X_train, y_train, data_handling, random_state=random_state)
        selector_fold = get_selector(selector_name, k=k)
        selector_fold.fit(X_train_proc, y_train_proc)

        selected_fold, _ = get_selected_features_from_selector(selector_fold, X_train_proc.columns)

        freq_counter.loc[selected_fold] += 1

        for feat in selected_fold:
            fold_details.append({
                "fold": fold_id,
                "miRNA": feat,
                "selected": 1
            })

    freq_df = freq_counter.reset_index()
    freq_df.columns = ["miRNA", "selection_count"]
    freq_df["selection_frequency"] = freq_df["selection_count"] / cv_splits
    freq_df = freq_df.sort_values(["selection_count", "miRNA"], ascending=[False, True]).reset_index(drop=True)
    freq_df.insert(0, "data_handling", data_handling)
    freq_df.insert(1, "optimization", optimization_status)
    freq_df.insert(2, "selector", selector_name)

    fold_detail_df = pd.DataFrame(fold_details)
    if not fold_detail_df.empty:
        fold_detail_df.insert(0, "data_handling", data_handling)
        fold_detail_df.insert(1, "optimization", optimization_status)
        fold_detail_df.insert(2, "selector", selector_name)

    return full_selected_df, freq_df, fold_detail_df

def save_top_frequency_plot(freq_df, title, out_path, top_n=30):
    plot_df = freq_df.head(top_n).copy()
    if plot_df.empty:
        return

    plt.figure(figsize=(12, 8))
    plt.barh(plot_df["miRNA"][::-1], plot_df["selection_count"][::-1])
    plt.xlabel("Selection count across CV folds")
    plt.ylabel("miRNA")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


# Classifierlar
def get_classifiers(optimization_status):
    classifiers_base = {
        "SVM": SVC(gamma='auto', probability=True, random_state=42),
        "Log. Reg.": LogisticRegression(solver='liblinear', random_state=42),
        "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42),
        "LDA": LinearDiscriminantAnalysis(),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "SGD": SGDClassifier(loss='log_loss', random_state=42),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42)
    }

    if optimization_status == "Unoptimized":
        return classifiers_base

    optimized_classifiers = {}
    for name, classifier in classifiers_base.items():
        optimized_classifiers[name] = classifier
    return optimized_classifiers


# Skorlama
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1_score': make_scorer(f1_score, zero_division=0),
    'mcc_score': make_scorer(matthews_corrcoef),
    'precision_score': make_scorer(precision_score, zero_division=0),
    'recall_score': make_scorer(recall_score, zero_division=0),
    'roc_auc_score': safe_roc_auc_estimator
}


def calculate_fold_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = _get_scores_from_estimator(model, X_test)

    try:
        auc = roc_auc_score(y_test, y_score)
        if np.isnan(auc):
            auc = 0.5
        elif auc == 0.0:
            auc = EPS
    except Exception:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "ROC AUC": float(auc)
    }


# Pipeline kurma
def build_pipeline(classifier, selector_name, data_handling, optimization_status, k=30):
    selector = get_selector(selector_name, k=k)

    if data_handling == "Oversampled (SMOTE)":
        base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])
    elif data_handling == "Downsampled":
        base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('downsampler', RandomUnderSampler(random_state=42)),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])
    else:
        base_pipeline = SkPipeline([
            ('scaler', StandardScaler()),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])

    return base_pipeline


# classifierları GridSearch ile optimize ettim
def build_optimized_model(classifier_name, classifier, selector_name, data_handling, k=30):
    pipeline = build_pipeline(
        classifier=classifier,
        selector_name=selector_name,
        data_handling=data_handling,
        optimization_status="Optimized",
        k=k
    )

    grid = param_grids[classifier_name]

    inner_cv = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        cv=inner_cv,
        scoring='f1',
        n_jobs=1,
        error_score=0.0
    )
    return gs


# Ana değerlendirme fonksiyonu
def evaluate_dataset(X, y, groups, title, data_handling, selector_name, optimization_status, k=30):
    print(f"\n--- {title} | FS: {selector_name} | Opt: {optimization_status} ---")

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)

    classifiers = get_classifiers(optimization_status)
    outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}

    for clf_name, clf in classifiers.items():
        try:
            if optimization_status == "Optimized":
                fold_metrics = []

                for fold_id, (train_idx, test_idx) in enumerate(
                    outer_cv.split(X, y, groups),
                    start=1
                ):
                    X_train = X.iloc[train_idx].reset_index(drop=True)
                    X_test = X.iloc[test_idx].reset_index(drop=True)
                    y_train = y.iloc[train_idx].reset_index(drop=True)
                    y_test = y.iloc[test_idx].reset_index(drop=True)
                    groups_train = groups.iloc[train_idx].reset_index(drop=True)
                    groups_test = groups.iloc[test_idx].reset_index(drop=True)

                    overlap = set(groups_train).intersection(set(groups_test))
                    if overlap:
                        raise ValueError(
                            f"Fold {fold_id}: train ve test arasında "
                            f"{len(overlap)} ortak hasta bulundu."
                        )

                    model = build_optimized_model(
                        classifier_name=clf_name,
                        classifier=clf,
                        selector_name=selector_name,
                        data_handling=data_handling,
                        k=k
                    )

                    model.fit(
                        X_train,
                        y_train,
                        groups=groups_train
                    )

                    fold_metrics.append(
                        calculate_fold_metrics(
                            model=model,
                            X_test=X_test,
                            y_test=y_test
                        )
                    )

                fold_metrics_df = pd.DataFrame(fold_metrics)

                results[clf_name] = {
                    "Accuracy": fold_metrics_df["Accuracy"].mean(),
                    "F1-Score": fold_metrics_df["F1-Score"].mean(),
                    "MCC": fold_metrics_df["MCC"].mean(),
                    "Precision": fold_metrics_df["Precision"].mean(),
                    "Recall": fold_metrics_df["Recall"].mean(),
                    "ROC AUC": fold_metrics_df["ROC AUC"].mean()
                }

            else:
                model = build_pipeline(
                    classifier=clf,
                    selector_name=selector_name,
                    data_handling=data_handling,
                    optimization_status=optimization_status,
                    k=k
                )

                cv_results = cross_validate(
                    model,
                    X,
                    y,
                    groups=groups,
                    cv=outer_cv,
                    scoring=scoring,
                    n_jobs=-1,
                    error_score=0.0
                )

                results[clf_name] = {
                    "Accuracy": cv_results['test_accuracy'].mean(),
                    "F1-Score": cv_results['test_f1_score'].mean(),
                    "MCC": cv_results['test_mcc_score'].mean(),
                    "Precision": cv_results['test_precision_score'].mean(),
                    "Recall": cv_results['test_recall_score'].mean(),
                    "ROC AUC": cv_results['test_roc_auc_score'].mean()
                }

        except Exception as e:
            print(f"Hata ({clf_name}): {e}")
            results[clf_name] = {
                "Accuracy": 0.0,
                "F1-Score": 0.0,
                "MCC": 0.0,
                "Precision": 0.0,
                "Recall": 0.0,
                "ROC AUC": 0.0
            }

    results_df = pd.DataFrame(results).T.round(3)
    print(results_df)
    return results_df


# Downsample helper
def create_downsampled_df(df, label_col='label', majority_class=1, minority_class=0, random_state=42):
    df_majority = df[df[label_col] == majority_class]
    df_minority = df[df[label_col] == minority_class]

    n_min = len(df_minority)
    df_down = pd.concat([
        df_majority.sample(n=n_min, random_state=random_state),
        df_minority
    ]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df_down


# Ana blok
try:
    os.makedirs(PLOT_DIR, exist_ok=True)

    df = pd.read_csv(file_path)

    if 'label' not in df.columns:
        raise KeyError("Veri setinde 'label' kolonu bulunamadı.")

    if 'Sample' not in df.columns:
        raise KeyError("Veri setinde 'Sample' kolonu bulunamadı.")

    # ============================================================
    # NORMALİZASYON BLOĞU (raw count -> RPM -> log2(RPM+1))
    # ============================================================
    feature_cols = [
        col for col in df.columns
        if col not in ['label', 'Sample']
    ]

    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    df[feature_cols] = df[feature_cols].clip(lower=0)

    lib_sizes = df[feature_cols].sum(axis=1)
    lib_sizes = lib_sizes.replace(0, np.nan)

    rpm = df[feature_cols].div(lib_sizes, axis=0) * 1e6
    rpm = rpm.fillna(0.0)

    log_rpm = np.log2(rpm + 1.0)

    # ============================================================
    # LOW-EXPRESSION FILTERING
    # En az %20 sample'da RPM > 1 olan miRNA'ları tut
    # ============================================================
    min_sample_fraction = 0.20
    min_samples_required = int(np.ceil(rpm.shape[0] * min_sample_fraction))

    keep_mask = (rpm > 1).sum(axis=0) >= min_samples_required
    kept_features = keep_mask[keep_mask].index.tolist()

    if len(kept_features) == 0:
        raise ValueError("Low-expression filtering sonrası hiç miRNA kalmadı. Eşiği gevşetmelisin.")

    print(f"Toplam başlangıç feature sayısı: {len(feature_cols)}")
    print(f"Filtre sonrası kalan feature sayısı: {len(kept_features)}")
    print(f"Eşik: en az {min_samples_required} sample'da RPM > 1")

    df = pd.concat(
        [
            df[['Sample', 'label']].reset_index(drop=True),
            log_rpm[kept_features].reset_index(drop=True)
        ],
        axis=1
    )

    groups_full = df['Sample'].astype(str).str[:12].reset_index(drop=True)
    X_full = df.drop(['Sample', 'label'], axis=1).reset_index(drop=True)
    y_full = df['label'].reset_index(drop=True)

    all_results = []

    # 1) Unbalanced
    print("\n" + "=" * 60)
    print("STARTING UNBALANCED DATA ANALYSIS")
    print("=" * 60)

    for selector_name in ['ANOVA', 'MI', 'mRMR']:
        for opt_status in ['Unoptimized', 'Optimized']:
            res = evaluate_dataset(
                X=X_full,
                y=y_full,
                groups=groups_full,
                title="Unbalanced Data",
                data_handling="Unbalanced",
                selector_name=selector_name,
                optimization_status=opt_status,
                k=K_FEATURES
            )
            res['Data Handling'] = 'Unbalanced'
            res['Feature Selector'] = selector_name
            res['Optimization'] = opt_status
            all_results.append(res)

    # 2) Downsampled
    print("\n" + "=" * 60)
    print("STARTING DOWNSAMPLED DATA ANALYSIS")
    print("=" * 60)

    for selector_name in ['ANOVA', 'MI', 'mRMR']:
        for opt_status in ['Unoptimized', 'Optimized']:
            res = evaluate_dataset(
                X=X_full,
                y=y_full,
                groups=groups_full,
                title="Downsampled Data",
                data_handling="Downsampled",
                selector_name=selector_name,
                optimization_status=opt_status,
                k=K_FEATURES
            )
            res['Data Handling'] = 'Downsampled'
            res['Feature Selector'] = selector_name
            res['Optimization'] = opt_status
            all_results.append(res)

    # 3) SMOTE
    print("\n" + "=" * 60)
    print("STARTING SMOTE WITHIN-CV ANALYSIS")
    print("=" * 60)

    if y_full.value_counts().min() > 1:
        for selector_name in ['ANOVA', 'MI', 'mRMR']:
            for opt_status in ['Unoptimized', 'Optimized']:
                res = evaluate_dataset(
                    X=X_full,
                    y=y_full,
                    groups=groups_full,
                    title="Oversampled Data",
                    data_handling="Oversampled (SMOTE)",
                    selector_name=selector_name,
                    optimization_status=opt_status,
                    k=K_FEATURES
                )
                res['Data Handling'] = 'Oversampled (SMOTE)'
                res['Feature Selector'] = selector_name
                res['Optimization'] = opt_status
                all_results.append(res)
    else:
        print("SMOTE analizi atlandı: Azınlık sınıfı çok küçük.")

    # ============================================================
    # Seçilen feature'ları çıkar ve grafiklerini oluştur
    # ============================================================
    scenario_datasets = [
        ("Unbalanced", "Unoptimized", X_full, y_full),
        ("Unbalanced", "Optimized", X_full, y_full),
        ("Downsampled", "Unoptimized", X_full, y_full),
        ("Downsampled", "Optimized", X_full, y_full)
    ]

    if y_full.value_counts().min() > 1:
        scenario_datasets.extend([
            ("Oversampled (SMOTE)", "Unoptimized", X_full, y_full),
            ("Oversampled (SMOTE)", "Optimized", X_full, y_full)
        ])

    # EKLENDİ:
    # Her senaryonun hangi dataset ile çalıştığını sakla
    scenario_dataset_lookup = {
        ("Unbalanced", "Unoptimized"): (X_full, y_full),
        ("Unbalanced", "Optimized"): (X_full, y_full),
        ("Downsampled", "Unoptimized"): (X_full, y_full),
        ("Downsampled", "Optimized"): (X_full, y_full)
    }

    if y_full.value_counts().min() > 1:
        scenario_dataset_lookup.update({
            ("Oversampled (SMOTE)", "Unoptimized"): (X_full, y_full),
            ("Oversampled (SMOTE)", "Optimized"): (X_full, y_full)
        })

    scenario_feature_map = {}

    with pd.ExcelWriter(FEATURE_OUTPUT_EXCEL, engine='openpyxl') as feat_writer:
        for data_handling, optimization_status, X_use, y_use in scenario_datasets:
            for selector_name in ['ANOVA', 'MI', 'mRMR']:
                full_selected_df, freq_df, fold_detail_df = inspect_selected_features_for_scenario(
                    X=X_use,
                    y=y_use,
                    groups=groups_full,
                    data_handling=data_handling,
                    selector_name=selector_name,
                    optimization_status=optimization_status,
                    k=K_FEATURES,
                    cv_splits=5,
                    random_state=42
                )

                code = scenario_code(data_handling, optimization_status, selector_name)

                full_sheet = make_safe_sheet_name(f"{code}_full")
                freq_sheet = make_safe_sheet_name(f"{code}_freq")
                fold_sheet = make_safe_sheet_name(f"{code}_folds")

                full_selected_df.to_excel(feat_writer, sheet_name=full_sheet, index=False)
                freq_df.to_excel(feat_writer, sheet_name=freq_sheet, index=False)

                if not fold_detail_df.empty:
                    fold_detail_df.to_excel(feat_writer, sheet_name=fold_sheet, index=False)

                plot_file = os.path.join(PLOT_DIR, f"{code}_top_selected.png")
                plot_title = f"{data_handling} | {optimization_status} | {selector_name}\nTop miRNA selection frequency"
                save_top_frequency_plot(freq_df, plot_title, plot_file, top_n=30)

                scenario_feature_map[(data_handling, optimization_status, selector_name)] = {
                    "full_selected_df": full_selected_df.copy(),
                    "freq_df": freq_df.copy(),
                    "code": code
                }

    # ============================================================
    # Performans sonuçlarını yaz
    # ============================================================
    final_results_df = None
    if all_results:
        final_results_df = pd.concat(all_results).reset_index().rename(columns={'index': 'Classifier'})

        with pd.ExcelWriter(RESULTS_EXCEL, engine='openpyxl') as writer:
            all_sorted = final_results_df.sort_values(
                by=['Data Handling', 'Optimization', 'Feature Selector', 'MCC'],
                ascending=[True, True, True, False]
            )
            all_sorted.to_excel(writer, sheet_name=TOPSIS_SHEET_NAME, index=False)

            for handling_type in final_results_df['Data Handling'].unique():
                for opt_status in final_results_df['Optimization'].unique():
                    sheet_name = f"{handling_type}-{opt_status}"[:31]
                    subset_df = final_results_df[
                        (final_results_df['Data Handling'] == handling_type) &
                        (final_results_df['Optimization'] == opt_status)
                    ].copy()

                    subset_sorted = subset_df.sort_values(
                        by=['Feature Selector', 'MCC'],
                        ascending=[True, False]
                    )
                    subset_sorted.to_excel(writer, sheet_name=sheet_name, index=False)

    # ============================================================
    # TOPSIS Analizi
    # ============================================================
    if final_results_df is not None and not final_results_df.empty:
        final_ranked_results = topsis_analysis(final_results_df, METRICS_TO_EVALUATE, METRIC_WEIGHTS)

        optimized_only = final_ranked_results[final_ranked_results["Optimization"] == "Optimized"].copy()
        best_optimized_row = None
        best_optimized_features_df = pd.DataFrame()
        best_optimized_freq_df = pd.DataFrame()
        best_optimized_summary_df = pd.DataFrame()
        best_optimized_biomarker_df = pd.DataFrame()

        if not optimized_only.empty:
            best_optimized_row = optimized_only.sort_values("Rank", ascending=True).iloc[0]

            best_key = (
                best_optimized_row["Data Handling"],
                best_optimized_row["Optimization"],
                best_optimized_row["Feature Selector"]
            )

            if best_key in scenario_feature_map:
                best_optimized_features_df = scenario_feature_map[best_key]["full_selected_df"].copy()
                best_optimized_freq_df = scenario_feature_map[best_key]["freq_df"].copy()

                # EKLENDİ:
                # En iyi optimized senaryonun seçtiği miRNA'ların yönünü hesapla
                selected_mirnas = best_optimized_features_df["miRNA"].tolist()

                # Ekspresyon yönü tüm orijinal normalize edilmiş veri üzerinden hesaplanır.
                # Downsampling yalnızca model eğitimi için kullanılır.
                best_optimized_biomarker_df = get_biomarker_direction_for_selected_mirnas(
                    X=X_full,
                    y=y_full,
                    selected_mirnas=selected_mirnas,
                    tumor_label=1,
                    normal_label=0
                )

                # Aynı feature tablosuna biomarker yön bilgisini ekle
                if not best_optimized_biomarker_df.empty:
                    best_optimized_features_df = best_optimized_features_df.merge(
                        best_optimized_biomarker_df,
                        on="miRNA",
                        how="left"
                    )

                best_optimized_summary_df = pd.DataFrame([{
                    "Best_Optimized_Classifier": best_optimized_row["Classifier"],
                    "Best_Optimized_Data_Handling": best_optimized_row["Data Handling"],
                    "Best_Optimized_Optimization": best_optimized_row["Optimization"],
                    "Best_Optimized_Feature_Selector": best_optimized_row["Feature Selector"],
                    "Best_Optimized_Topsis_Score": best_optimized_row["Topsis_Score"],
                    "Best_Optimized_Rank": best_optimized_row["Rank"],
                    "MCC": best_optimized_row["MCC"],
                    "F1-Score": best_optimized_row["F1-Score"],
                    "ROC AUC": best_optimized_row["ROC AUC"],
                    "Accuracy": best_optimized_row["Accuracy"],
                    "Precision": best_optimized_row["Precision"],
                    "Recall": best_optimized_row["Recall"]
                }])

        with pd.ExcelWriter(TOPSIS_OUTPUT_EXCEL, engine='openpyxl') as topsis_writer:
            final_ranked_results.to_excel(topsis_writer, sheet_name="TOPSIS_Sirali_30", index=False)

            if not optimized_only.empty:
                optimized_only.to_excel(topsis_writer, sheet_name="TOPSIS_Optimized_30", index=False)

            if not best_optimized_summary_df.empty:
                best_optimized_summary_df.to_excel(topsis_writer, sheet_name="En_Iyi_Optimized_30", index=False)

            if not best_optimized_features_df.empty:
                best_optimized_features_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_miRNA_30",
                    index=False
                )

            if not best_optimized_freq_df.empty:
                best_optimized_freq_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_Freq_30",
                    index=False
                )

            # EKLENDİ:
            # Biomarker yön bilgisini ayrı sheet olarak da kaydet
            if not best_optimized_biomarker_df.empty:
                best_optimized_biomarker_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_Biomarker_30",
                    index=False
                )

        print("\n" + "=" * 60)
        print("🏆 TOPSIS ANALİZİ BAŞARIYLA TAMAMLANDI 🏆")
        print("=" * 60)
        print("\nEn iyi 10 model kombinasyonu:\n")
        print(final_ranked_results.head(10))

        if best_optimized_row is not None:
            print("\nEn iyi OPTIMIZED senaryo:")
            print(
                f"Classifier={best_optimized_row['Classifier']} | "
                f"Data Handling={best_optimized_row['Data Handling']} | "
                f"Selector={best_optimized_row['Feature Selector']} | "
                f"Topsis Score={best_optimized_row['Topsis_Score']:.4f}"
            )
            print(f"Bu senaryonun seçtiği miRNA'lar '{TOPSIS_OUTPUT_EXCEL}' içindeki 'En_Iyi_Optimized_miRNA' sayfasına yazıldı.")
            print(f"Biyomarker yön bilgileri '{TOPSIS_OUTPUT_EXCEL}' içindeki 'En_Iyi_Optimized_Biomarker' sayfasına yazıldı.")

    print("\n" + "=" * 60)
    print(f"TÜM SONUÇLAR BAŞARIYLA '{RESULTS_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print(f"SEÇİLEN FEATURE'LAR BAŞARIYLA '{FEATURE_OUTPUT_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print(f"GRAFİKLER '{PLOT_DIR}' klasörüne kaydedildi.")
    print(f"TOPSIS SONUÇLARI BAŞARIYLA '{TOPSIS_OUTPUT_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print("=" * 60)

except FileNotFoundError:
    print(f"❌ HATA: '{file_path}' veya ara çıktı dosyalarından biri bulunamadı.")
except KeyError as e:
    print(f"❌ HATA: {e}")
    
except ValueError as e:
    print(f"❌ HATA: {e}")
except Exception as e:
    print(f"Beklenmedik bir hata oluştu: {e}")

Toplam başlangıç feature sayısı: 1881
Filtre sonrası kalan feature sayısı: 415
Eşik: en az 113 sample'da RPM > 1

STARTING UNBALANCED DATA ANALYSIS

--- Unbalanced Data | FS: ANOVA | Opt: Unoptimized ---
                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.986     0.992  0.926      0.996   0.988    0.989
Log. Reg.             0.982     0.990  0.911      0.996   0.984    0.995
Extra Trees           0.984     0.991  0.917      0.994   0.988    0.997
LDA                   0.980     0.989  0.904      0.998   0.980    0.994
Random Forest         0.984     0.991  0.916      0.988   0.994    0.996
Gradient Boosting     0.977     0.987  0.878      0.984   0.990    0.988
SGD                   0.975     0.986  0.878      0.988   0.984    0.977
KNN                   0.979     0.988  0.897      0.998   0.978    0.987
Decision Tree         0.974     0.985  0.858      0.984   0.986    0.927
AdaBoost              0.986     0.992  0.929      0.992   0.992   

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.77it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.989     0.994  0.942      0.996   0.992    0.988
Log. Reg.             0.988     0.993  0.937      0.996   0.990    0.996
Extra Trees           0.989     0.994  0.943      0.994   0.994    0.996
LDA                   0.984     0.991  0.921      0.998   0.984    0.995
Random Forest         0.984     0.991  0.915      0.990   0.992    0.995
Gradient Boosting     0.977     0.987  0.879      0.983   0.992    0.996
SGD                   0.979     0.988  0.891      0.986   0.990    0.988
KNN                   0.984     0.991  0.922      0.998   0.984    0.987
Decision Tree         0.970     0.983  0.839      0.980   0.986    0.911
AdaBoost              0.991     0.995  0.954      0.996   0.994    0.997

STARTING DOWNSAMPLED DATA ANALYSIS

--- Downsampled Data | FS: ANOVA | Opt: Unoptimized ---
                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.970    

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:04<00:00,  6.39it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.975     0.986  0.879      0.996   0.976    0.995
Log. Reg.             0.933     0.961  0.748      0.996   0.929    0.983
Extra Trees           0.954     0.974  0.812      0.998   0.951    0.996
LDA                   0.945     0.968  0.789      0.998   0.941    0.992
Random Forest         0.951     0.971  0.813      0.998   0.947    0.994
Gradient Boosting     0.876     0.926  0.603      0.992   0.870    0.961
SGD                   0.952     0.973  0.799      0.996   0.951    0.991
KNN                   0.945     0.968  0.791      0.998   0.941    0.974
Decision Tree         0.864     0.918  0.579      0.991   0.856    0.894
AdaBoost              0.938     0.964  0.775      0.998   0.933    0.994

STARTING SMOTE WITHIN-CV ANALYSIS

--- Oversampled Data | FS: ANOVA | Opt: Unoptimized ---
                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.986     

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.12it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.984     0.991  0.917      0.996   0.986    0.986
Log. Reg.             0.982     0.990  0.911      0.996   0.984    0.993
Extra Trees           0.981     0.989  0.900      0.992   0.986    0.995
LDA                   0.977     0.987  0.890      0.998   0.976    0.994
Random Forest         0.979     0.988  0.890      0.988   0.988    0.987
Gradient Boosting     0.981     0.989  0.900      0.992   0.986    0.996
SGD                   0.982     0.990  0.912      0.998   0.982    0.988
KNN                   0.975     0.986  0.884      0.998   0.974    0.981
Decision Tree         0.965     0.980  0.810      0.980   0.980    0.908
AdaBoost              0.981     0.989  0.902      0.994   0.984    0.996


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.43it/s]


--- TOPSIS Analizi Başlatılıyor ---

🏆 TOPSIS ANALİZİ BAŞARIYLA TAMAMLANDI 🏆

En iyi 10 model kombinasyonu:

          Classifier  Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC  \
0           AdaBoost     0.995     0.997  0.974      0.998   0.996    0.997   
1           AdaBoost     0.991     0.995  0.954      0.996   0.994    0.997   
2        Extra Trees     0.989     0.994  0.943      0.994   0.994    0.996   
3           AdaBoost     0.989     0.994  0.943      0.994   0.994    0.995   
4                SVM     0.989     0.994  0.942      0.996   0.992    0.988   
5        Extra Trees     0.988     0.993  0.938      0.994   0.992    0.997   
6          Log. Reg.     0.988     0.993  0.937      0.996   0.990    0.996   
7  Gradient Boosting     0.988     0.993  0.935      0.996   0.990    0.994   
8                SVM     0.988     0.993  0.935      0.996   0.990    0.989   
9        Extra Trees     0.988     0.993  0.934      0.994   0.992    0.996   

         Data Handlin

In [2]:
# ============================================================
# PUBLICATION FIGURES FROM SAVED RESULTS
# MODEL YENİDEN ÇALIŞTIRILMAZ
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# ============================================================
# 0. DOSYALAR
# ============================================================

TOPSIS_FILE = "topsis_sonuclari_sirali_30.xlsx"
FEATURE_FILE = "selected_features_by_scenario_30.xlsx"
RESULTS_FILE = "classification_results_leakage_fixed_30.xlsx"

OUT_DIR = "publication_figures_from_saved_results"

os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# FIGURE SAVE FUNCTION
# PNG + PDF oluşturur
# ============================================================

def save_fig(fig, filename_without_extension):

    png_path = os.path.join(
        OUT_DIR,
        filename_without_extension + ".png"
    )

    pdf_path = os.path.join(
        OUT_DIR,
        filename_without_extension + ".pdf"
    )

    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.close(fig)

    print("Kaydedildi:")
    print(" ", png_path)
    print(" ", pdf_path)


# ============================================================
# DOSYALAR VAR MI?
# ============================================================

for f in [TOPSIS_FILE, FEATURE_FILE, RESULTS_FILE]:

    if os.path.exists(f):
        print("✓ Bulundu:", f)

    else:
        print("⚠ Bulunamadı:", f)


# ============================================================
# SHEET KONTROLÜ
# ============================================================

topsis_xls = pd.ExcelFile(TOPSIS_FILE)
feature_xls = pd.ExcelFile(FEATURE_FILE)

print("\nTOPSIS sheets:")
print(topsis_xls.sheet_names)

print("\nFeature sheets:")
print(feature_xls.sheet_names)


# ============================================================
# DOSYALARI OKU
# SENİN ESKİ DOSYANDAKİ GERÇEK SHEET İSİMLERİ
# ============================================================

topsis_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="TOPSIS_Sirali"
)

optimized_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="TOPSIS_Optimized"
)

summary_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized"
)

final_panel_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_miRNA"
)

freq_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_Freq"
)

bio_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_Biomarker"
)


print("\nEn iyi optimized model:")
display(summary_df)

print("\nFinal panel ilk satırlar:")
display(final_panel_df.head())


# ============================================================
# 1. TOP 10 TOPSIS MODEL CONFIGURATIONS
# ============================================================

top10 = (
    topsis_df
    .sort_values("Rank", ascending=True)
    .head(10)
    .copy()
)

top10["Model"] = (
    top10["Classifier"].astype(str)
    + " | "
    + top10["Data Handling"].astype(str)
    + " | "
    + top10["Feature Selector"].astype(str)
    + " | "
    + top10["Optimization"].astype(str)
)

plot_df = top10.sort_values(
    "Topsis_Score",
    ascending=True
)

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    plot_df["Model"],
    plot_df["Topsis_Score"]
)

ax.set_xlabel(
    "TOPSIS score",
    fontsize=11
)

ax.set_ylabel("")

ax.set_title(
    "Top 10 model configurations according to TOPSIS",
    fontsize=13
)

for i, value in enumerate(plot_df["Topsis_Score"]):

    ax.text(
        value + 0.004,
        i,
        f"{value:.3f}",
        va="center",
        fontsize=8
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

save_fig(
    fig,
    "Fig1_TOPSIS_top10_models"
)


# ============================================================
# 2. CLASS-HANDLING MEAN TOPSIS
# ============================================================

strategy_df = (
    topsis_df
    .groupby(
        "Data Handling",
        as_index=False
    )["Topsis_Score"]
    .mean()
    .sort_values(
        "Topsis_Score",
        ascending=False
    )
)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    strategy_df["Data Handling"],
    strategy_df["Topsis_Score"]
)

ax.set_ylabel(
    "Mean TOPSIS score",
    fontsize=11
)

ax.set_xlabel(
    "Class-handling strategy",
    fontsize=11
)

ax.set_title(
    "Comparison of class-handling strategies",
    fontsize=13
)

ax.tick_params(
    axis="x",
    rotation=15
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, value in zip(
    bars,
    strategy_df["Topsis_Score"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

fig.tight_layout()

save_fig(
    fig,
    "Fig2_class_handling_mean_TOPSIS"
)

print("\nMean TOPSIS by class handling:")
display(strategy_df)


# ============================================================
# 3. FINAL PANEL SELECTION STABILITY
# ============================================================

final_mirnas = (
    final_panel_df["miRNA"]
    .dropna()
    .astype(str)
    .tolist()
)

stability_df = freq_df[
    freq_df["miRNA"]
    .astype(str)
    .isin(final_mirnas)
].copy()

stability_df = stability_df.sort_values(
    ["selection_count", "miRNA"],
    ascending=[True, True]
)

# Model summary'den selector ismini bul
if (
    "Best_Optimized_Feature_Selector"
    in summary_df.columns
):

    final_selector = str(
        summary_df.iloc[0][
            "Best_Optimized_Feature_Selector"
        ]
    )

else:
    final_selector = "mRMR"


fig, ax = plt.subplots(
    figsize=(10, 9)
)

ax.barh(
    stability_df["miRNA"],
    stability_df["selection_count"]
)

ax.set_xlim(0, 5.4)

ax.set_xticks(
    [0, 1, 2, 3, 4, 5]
)

ax.set_xlabel(
    "Selection count across five outer training folds",
    fontsize=11
)

ax.set_ylabel(
    "miRNA",
    fontsize=11
)

ax.set_title(
    f"Selection stability of the final {final_selector} miRNA panel",
    fontsize=13
)

for i, value in enumerate(
    stability_df["selection_count"]
):

    ax.text(
        value + 0.06,
        i,
        f"{int(value)}/5",
        va="center",
        fontsize=8
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

save_fig(
    fig,
    "Fig3_final_panel_selection_stability"
)

print("\nFinal panel selection stability:")

display(
    stability_df[
        [
            "miRNA",
            "selection_count",
            "selection_frequency"
        ]
    ]
)


# ============================================================
# 4. FINAL 30-miRNA EXPRESSION DIFFERENCES
# ============================================================

bio_plot_df = bio_df.copy()

bio_plot_df[
    "log2FC_Tumor_vs_Normal"
] = pd.to_numeric(
    bio_plot_df[
        "log2FC_Tumor_vs_Normal"
    ],
    errors="coerce"
)

bio_plot_df = (
    bio_plot_df
    .dropna(
        subset=[
            "log2FC_Tumor_vs_Normal"
        ]
    )
    .sort_values(
        "log2FC_Tumor_vs_Normal",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(10, 10)
)

ax.barh(
    bio_plot_df["miRNA"],
    bio_plot_df[
        "log2FC_Tumor_vs_Normal"
    ]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Mean log2(RPM+1) difference (tumor - normal)",
    fontsize=11
)

ax.set_ylabel(
    "miRNA",
    fontsize=11
)

ax.set_title(
    "Expression differences of the final 30-miRNA panel",
    fontsize=13
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

save_fig(
    fig,
    "Fig4_expression_difference_final_30"
)


# ============================================================
# 5. ANOVA - MI - mRMR VENN DIAGRAM
# Unbalanced + Optimized senaryosu
# ============================================================

anova_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_ANOVA_full"
)

mi_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_MI_full"
)

mrmr_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_mRMR_full"
)

A = set(
    anova_df["miRNA"]
    .dropna()
    .astype(str)
)

B = set(
    mi_df["miRNA"]
    .dropna()
    .astype(str)
)

C = set(
    mrmr_df["miRNA"]
    .dropna()
    .astype(str)
)


only_a = len(
    A - B - C
)

only_b = len(
    B - A - C
)

only_c = len(
    C - A - B
)

ab = len(
    (A & B) - C
)

ac = len(
    (A & C) - B
)

bc = len(
    (B & C) - A
)

abc = len(
    A & B & C
)


fig, ax = plt.subplots(
    figsize=(8, 8)
)


# circles
ax.add_patch(
    Circle(
        (0.40, 0.58),
        0.28,
        fill=False,
        linewidth=2
    )
)

ax.add_patch(
    Circle(
        (0.60, 0.58),
        0.28,
        fill=False,
        linewidth=2
    )
)

ax.add_patch(
    Circle(
        (0.50, 0.40),
        0.28,
        fill=False,
        linewidth=2
    )
)


# labels
ax.text(
    0.18,
    0.84,
    "ANOVA",
    fontsize=13
)

ax.text(
    0.72,
    0.84,
    "MI",
    fontsize=13
)

ax.text(
    0.46,
    0.07,
    "mRMR",
    fontsize=13
)


# values
ax.text(
    0.24,
    0.63,
    str(only_a),
    fontsize=13,
    ha="center"
)

ax.text(
    0.76,
    0.63,
    str(only_b),
    fontsize=13,
    ha="center"
)

ax.text(
    0.50,
    0.18,
    str(only_c),
    fontsize=13,
    ha="center"
)

ax.text(
    0.50,
    0.72,
    str(ab),
    fontsize=13,
    ha="center"
)

ax.text(
    0.35,
    0.40,
    str(ac),
    fontsize=13,
    ha="center"
)

ax.text(
    0.65,
    0.40,
    str(bc),
    fontsize=13,
    ha="center"
)

ax.text(
    0.50,
    0.51,
    str(abc),
    fontsize=16,
    fontweight="bold",
    ha="center"
)


ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_aspect(
    "equal"
)

ax.axis(
    "off"
)

ax.set_title(
    "Overlap among top 30 miRNAs selected by ANOVA, MI, and mRMR",
    fontsize=13
)

save_fig(
    fig,
    "Fig5_Venn_ANOVA_MI_mRMR"
)


print(
    "\nANOVA ∩ MI ∩ mRMR ortak miRNA sayısı:",
    abc
)

print(
    "\nÜç yöntemde ortak olan miRNA'lar:"
)

print(
    sorted(
        A & B & C
    )
)


# ============================================================
# 6. BEST OPTIMIZED MODEL PERFORMANCE
# ============================================================

row = summary_df.iloc[0]

metrics = [
    "Accuracy",
    "F1-Score",
    "MCC",
    "Precision",
    "Recall",
    "ROC AUC"
]

values = [
    float(row[m])
    for m in metrics
]

fig, ax = plt.subplots(
    figsize=(8, 5)
)

bars = ax.bar(
    metrics,
    values
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Score",
    fontsize=11
)

ax.set_title(
    "Performance of the highest-ranking optimized model",
    fontsize=13
)

ax.tick_params(
    axis="x",
    rotation=30
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


for bar, value in zip(
    bars,
    values
):

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,

        value + 0.01,

        f"{value:.3f}",

        ha="center",
        va="bottom",
        fontsize=9
    )


fig.tight_layout()

save_fig(
    fig,
    "Fig6_best_optimized_model_performance"
)


# ============================================================
# 7. KONTROL ÖZETİ
# ============================================================

print("\n" + "=" * 70)

print(
    "TÜM KAYITLI SONUÇ GRAFİKLERİ "
    "BAŞARIYLA OLUŞTURULDU"
)

print("=" * 70)

print(
    "\nGrafik klasörü:"
)

print(
    os.path.abspath(OUT_DIR)
)

print(
    "\nFinal panel miRNA sayısı:",
    len(final_mirnas)
)

print(
    "Venn - üç yöntemde ortak miRNA:",
    abc
)

print(
    "\nEn iyi optimized model:"
)

display(
    summary_df
)

print("=" * 70)

✓ Bulundu: topsis_sonuclari_sirali_30.xlsx
✓ Bulundu: selected_features_by_scenario_30.xlsx
✓ Bulundu: classification_results_leakage_fixed_30.xlsx

TOPSIS sheets:
['TOPSIS_Sirali', 'TOPSIS_Optimized', 'En_Iyi_Optimized', 'En_Iyi_Optimized_miRNA', 'En_Iyi_Optimized_Freq', 'En_Iyi_Optimized_Biomarker']

Feature sheets:
['UB_U_ANOVA_full', 'UB_U_ANOVA_freq', 'UB_U_ANOVA_folds', 'UB_U_MI_full', 'UB_U_MI_freq', 'UB_U_MI_folds', 'UB_U_mRMR_full', 'UB_U_mRMR_freq', 'UB_U_mRMR_folds', 'UB_O_ANOVA_full', 'UB_O_ANOVA_freq', 'UB_O_ANOVA_folds', 'UB_O_MI_full', 'UB_O_MI_freq', 'UB_O_MI_folds', 'UB_O_mRMR_full', 'UB_O_mRMR_freq', 'UB_O_mRMR_folds', 'DS_U_ANOVA_full', 'DS_U_ANOVA_freq', 'DS_U_ANOVA_folds', 'DS_U_MI_full', 'DS_U_MI_freq', 'DS_U_MI_folds', 'DS_U_mRMR_full', 'DS_U_mRMR_freq', 'DS_U_mRMR_folds', 'DS_O_ANOVA_full', 'DS_O_ANOVA_freq', 'DS_O_ANOVA_folds', 'DS_O_MI_full', 'DS_O_MI_freq', 'DS_O_MI_folds', 'DS_O_mRMR_full', 'DS_O_mRMR_freq', 'DS_O_mRMR_folds', 'OS_U_ANOVA_full', 'OS_U_ANOVA_

,Best_Optimized_Classifier,Best_Optimized_Data_Handling,Best_Optimized_Optimization,Best_Optimized_Feature_Selector,Best_Optimized_Topsis_Score,Best_Optimized_Rank,MCC,F1-Score,ROC AUC,Accuracy,Precision,Recall
0,AdaBoost,Unbalanced,Optimized,mRMR,0.950669,2,0.954,0.995,0.997,0.991,0.996,0.994



Final panel ilk satırlar:


,data_handling,optimization,selector,miRNA,score_or_rank,Tumor_mean,Normal_mean,log2FC_Tumor_vs_Normal,Direction,Biomarker_Type,Interpretation
0,Unbalanced,Optimized,mRMR,hsa-mir-1258,1,0.355844,1.687478,-1.331634,Down in Tumor,Negative biomarker candidate,Lower expression in tumor than normal
1,Unbalanced,Optimized,mRMR,hsa-mir-4473,2,0.777377,1.090949,-0.313571,Down in Tumor,Negative biomarker candidate,Lower expression in tumor than normal
2,Unbalanced,Optimized,mRMR,hsa-mir-34a,3,8.749259,6.823956,1.925303,Up in Tumor,Positive biomarker candidate,Higher expression in tumor than normal
3,Unbalanced,Optimized,mRMR,hsa-mir-144,4,7.139250,9.756618,-2.617368,Down in Tumor,Negative biomarker candidate,Lower expression in tumor than normal
4,Unbalanced,Optimized,mRMR,hsa-mir-4709,5,3.357682,1.273346,2.084337,Up in Tumor,Positive biomarker candidate,Higher expression in tumor than normal


Kaydedildi:
  publication_figures_from_saved_results\Fig1_TOPSIS_top10_models.png
  publication_figures_from_saved_results\Fig1_TOPSIS_top10_models.pdf
Kaydedildi:
  publication_figures_from_saved_results\Fig2_class_handling_mean_TOPSIS.png
  publication_figures_from_saved_results\Fig2_class_handling_mean_TOPSIS.pdf

Mean TOPSIS by class handling:


,Data Handling,Topsis_Score
2,Unbalanced,0.821003
1,Oversampled (SMOTE),0.788375
0,Downsampled,0.459600


Kaydedildi:
  publication_figures_from_saved_results\Fig3_final_panel_selection_stability.png
  publication_figures_from_saved_results\Fig3_final_panel_selection_stability.pdf

Final panel selection stability:


,miRNA,selection_count,selection_frequency
285,hsa-mir-4473,0,0.0
35,hsa-mir-218-2,1,0.2
30,hsa-mir-138-2,2,0.4
24,hsa-mir-181a-2,3,0.6
25,hsa-mir-181b-2,3,0.6
27,hsa-mir-363,3,0.6
23,hsa-mir-28,4,0.8
0,hsa-mir-100,5,1.0
1,hsa-mir-1179,5,1.0
2,hsa-mir-1258,5,1.0


Kaydedildi:
  publication_figures_from_saved_results\Fig4_expression_difference_final_30.png
  publication_figures_from_saved_results\Fig4_expression_difference_final_30.pdf
Kaydedildi:
  publication_figures_from_saved_results\Fig5_Venn_ANOVA_MI_mRMR.png
  publication_figures_from_saved_results\Fig5_Venn_ANOVA_MI_mRMR.pdf

ANOVA ∩ MI ∩ mRMR ortak miRNA sayısı: 22

Üç yöntemde ortak olan miRNA'lar:
['hsa-mir-1179', 'hsa-mir-1258', 'hsa-mir-144', 'hsa-mir-146b', 'hsa-mir-181a-2', 'hsa-mir-181b-2', 'hsa-mir-221', 'hsa-mir-222', 'hsa-mir-26a-2', 'hsa-mir-28', 'hsa-mir-34a', 'hsa-mir-374b', 'hsa-mir-451a', 'hsa-mir-4709', 'hsa-mir-4732', 'hsa-mir-486-1', 'hsa-mir-486-2', 'hsa-mir-652', 'hsa-mir-873', 'hsa-mir-9-1', 'hsa-mir-9-2', 'hsa-mir-9-3']
Kaydedildi:
  publication_figures_from_saved_results\Fig6_best_optimized_model_performance.png
  publication_figures_from_saved_results\Fig6_best_optimized_model_performance.pdf

TÜM KAYITLI SONUÇ GRAFİKLERİ BAŞARIYLA OLUŞTURULDU

Grafik klasörü:
C:\

,Best_Optimized_Classifier,Best_Optimized_Data_Handling,Best_Optimized_Optimization,Best_Optimized_Feature_Selector,Best_Optimized_Topsis_Score,Best_Optimized_Rank,MCC,F1-Score,ROC AUC,Accuracy,Precision,Recall
0,AdaBoost,Unbalanced,Optimized,mRMR,0.950669,2,0.954,0.995,0.997,0.991,0.996,0.994


In [3]:
# =====================================================================
# TCGA-THCA miRNA STUDY
# FINAL PUBLICATION FIGURE GENERATOR
#
# Saved Excel results -> publication-quality figures
# PLUS:
# Final model only -> OOF confusion matrix + ROC curve
#
# Final model:
# Unbalanced + mRMR + AdaBoost + Optimized
# =====================================================================

import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_classif
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)
from mrmr import mrmr_classif

warnings.filterwarnings("ignore")


# =====================================================================
# 0. PATHS
# =====================================================================

# Eğer dosyaların başka klasördeyse SADECE burayı değiştir.
BASE_DIR = r"C:\Users\DURU VURAL\Desktop\proje"

CSV_NAME = "THCA_miRNA_unique_patients_only.csv"
TOPSIS_NAME = "topsis_sonuclari_sirali_30.xlsx"
FEATURE_NAME = "selected_features_by_scenario_30.xlsx"

OUT_DIR = os.path.join(
    BASE_DIR,
    "FINAL_PUBLICATION_FIGURES"
)

os.makedirs(OUT_DIR, exist_ok=True)


def resolve_file(filename):
    """
    Önce BASE_DIR altında arar.
    Bulamazsa notebook'un mevcut klasöründe arar.
    """
    p1 = os.path.join(BASE_DIR, filename)

    if os.path.exists(p1):
        return p1

    if os.path.exists(filename):
        return os.path.abspath(filename)

    raise FileNotFoundError(
        f"\nDosya bulunamadı: {filename}\n"
        f"Aranan yerler:\n"
        f"1) {p1}\n"
        f"2) {os.path.abspath(filename)}"
    )


CSV_FILE = resolve_file(CSV_NAME)
TOPSIS_FILE = resolve_file(TOPSIS_NAME)
FEATURE_FILE = resolve_file(FEATURE_NAME)


print("=" * 75)
print("FILES")
print("=" * 75)
print("CSV     :", CSV_FILE)
print("TOPSIS  :", TOPSIS_FILE)
print("FEATURE :", FEATURE_FILE)
print("OUTPUT  :", OUT_DIR)


# =====================================================================
# 1. PUBLICATION STYLE
# =====================================================================

# Renk paleti
NAVY = "#25364A"
TEAL = "#2A9D8F"
BLUE = "#457B9D"
LIGHT_BLUE = "#A8DADC"
GOLD = "#E9C46A"
ORANGE = "#F4A261"
RED = "#E76F51"
PALE_RED = "#F4B6A6"
PALE_TEAL = "#A8DCCF"
PALE_GOLD = "#F4E1A1"
GREY = "#68737D"
LIGHT_GREY = "#E7EBEF"
GRID = "#D9DEE3"
WHITE = "#FFFFFF"
BG = "#FAFBFC"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": NAVY,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.facecolor": WHITE,
    "axes.facecolor": WHITE,
    "savefig.facecolor": WHITE,
    "savefig.dpi": 600
})


def clean_axes(ax, grid_axis=None):
    """Consistent publication styling."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if grid_axis == "x":
        ax.grid(
            axis="x",
            linestyle="--",
            linewidth=0.6,
            alpha=0.45,
            color=GRID
        )
        ax.set_axisbelow(True)

    elif grid_axis == "y":
        ax.grid(
            axis="y",
            linestyle="--",
            linewidth=0.6,
            alpha=0.45,
            color=GRID
        )
        ax.set_axisbelow(True)


def save_fig(fig, name):
    """
    Her figürü hem PNG (600 dpi) hem PDF olarak kaydeder.
    """
    png = os.path.join(OUT_DIR, name + ".png")
    pdf = os.path.join(OUT_DIR, name + ".pdf")

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    print("✓", name)


# =====================================================================
# 2. LOAD SAVED RESULTS
# =====================================================================

topsis_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="TOPSIS_Sirali"
)

optimized_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="TOPSIS_Optimized"
)

summary_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized"
)

final_panel_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_miRNA"
)

freq_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_Freq"
)

bio_df = pd.read_excel(
    TOPSIS_FILE,
    sheet_name="En_Iyi_Optimized_Biomarker"
)


# =====================================================================
# 3. CRITICAL SAFETY CHECK
# Prevent accidental use of the newer ANOVA result set
# =====================================================================

best_classifier = str(
    summary_df.iloc[0]["Best_Optimized_Classifier"]
).strip()

best_handling = str(
    summary_df.iloc[0]["Best_Optimized_Data_Handling"]
).strip()

best_selector = str(
    summary_df.iloc[0]["Best_Optimized_Feature_Selector"]
).strip()


print("\n" + "=" * 75)
print("SAVED FINAL MODEL CHECK")
print("=" * 75)

print("Classifier    :", best_classifier)
print("Data handling :", best_handling)
print("Selector      :", best_selector)


expected_ok = (
    best_classifier.lower() == "adaboost"
    and best_handling.lower() == "unbalanced"
    and best_selector.lower() == "mrmr"
)

if not expected_ok:
    raise RuntimeError(
        "\nSTOP!\n"
        "Bu Excel dosyasında final optimized model "
        "'Unbalanced + mRMR + AdaBoost' değil.\n"
        "Yanlışlıkla yeni ANOVA sonuç dosyasını kullanıyor olabilirsin.\n"
        "Eski/orijinal sonuç Excel dosyasını seç."
    )

print("\n✓ Doğru eski/canonical sonuç seti bulundu.")


# =====================================================================
# 4. FIGURE 1
# TOP 10 TOPSIS CONFIGURATIONS
# =====================================================================

top10 = (
    topsis_df
    .sort_values("Rank", ascending=True)
    .head(10)
    .copy()
)

handling_short = {
    "Unbalanced": "Original",
    "Oversampled (SMOTE)": "SMOTE",
    "Downsampled": "Downsampled"
}

opt_short = {
    "Optimized": "Opt.",
    "Unoptimized": "Unopt."
}

top10["Label"] = top10.apply(
    lambda r:
        f"{r['Classifier']} | "
        f"{handling_short.get(r['Data Handling'], r['Data Handling'])} | "
        f"{r['Feature Selector']} | "
        f"{opt_short.get(r['Optimization'], r['Optimization'])}",
    axis=1
)

plot_df = top10.sort_values(
    "Topsis_Score",
    ascending=True
).reset_index(drop=True)

colors = []

for _, r in plot_df.iterrows():

    # Final optimized model
    if (
        r["Classifier"] == "AdaBoost"
        and r["Data Handling"] == "Unbalanced"
        and r["Feature Selector"] == "mRMR"
        and r["Optimization"] == "Optimized"
    ):
        colors.append(TEAL)

    # Overall rank #1
    elif int(r["Rank"]) == 1:
        colors.append(ORANGE)

    else:
        colors.append(LIGHT_BLUE)


fig, ax = plt.subplots(figsize=(12, 7.5))

bars = ax.barh(
    plot_df["Label"],
    plot_df["Topsis_Score"],
    color=colors,
    edgecolor="none"
)

ax.set_xlabel("TOPSIS score")
ax.set_ylabel("")
ax.set_title(
    "Top 10 machine-learning configurations by TOPSIS score",
    pad=15
)

xmin = max(
    0,
    float(plot_df["Topsis_Score"].min()) - 0.08
)

ax.set_xlim(
    xmin,
    1.03
)

clean_axes(ax, grid_axis="x")

for bar, (_, r) in zip(bars, plot_df.iterrows()):

    value = float(r["Topsis_Score"])

    ax.text(
        value + 0.006,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        fontsize=9,
        fontweight="bold",
        color=NAVY
    )

# legend-like explanation
ax.text(
    0.99,
    -0.12,
    "Orange: overall highest-ranked configuration   •   "
    "Teal: highest-ranked optimized configuration",
    transform=ax.transAxes,
    ha="right",
    fontsize=8.5,
    color=GREY
)

fig.tight_layout()

save_fig(
    fig,
    "Fig1_TOPSIS_top10"
)


# =====================================================================
# 5. FIGURE 2
# CLASS-HANDLING STRATEGIES
# =====================================================================

strategy = (
    topsis_df
    .groupby("Data Handling")["Topsis_Score"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

strategy["Display"] = strategy["Data Handling"].map({
    "Unbalanced": "Original\nunbalanced",
    "Oversampled (SMOTE)": "SMOTE",
    "Downsampled": "Random\nundersampling"
}).fillna(strategy["Data Handling"])


fig, ax = plt.subplots(figsize=(7.5, 5.5))

bars = ax.bar(
    strategy["Display"],
    strategy["mean"],
    yerr=strategy["std"],
    capsize=5,
    color=[TEAL, GOLD, LIGHT_BLUE][:len(strategy)],
    edgecolor="none",
    width=0.62
)

ax.set_ylabel("Mean TOPSIS score")
ax.set_xlabel("")
ax.set_title(
    "Model performance across class-handling strategies",
    pad=14
)

ax.set_ylim(
    0,
    min(
        1.05,
        max(strategy["mean"] + strategy["std"]) + 0.12
    )
)

clean_axes(ax, grid_axis="y")

for bar, value in zip(bars, strategy["mean"]):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.025,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color=NAVY
    )

ax.text(
    0.5,
    -0.15,
    "Bars show mean TOPSIS score; error bars show SD across model configurations.",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.5,
    color=GREY
)

fig.tight_layout()

save_fig(
    fig,
    "Fig2_class_handling_TOPSIS"
)


# =====================================================================
# 6. FIGURE 3
# FINAL PANEL SELECTION STABILITY
# =====================================================================

final_mirnas = (
    final_panel_df["miRNA"]
    .dropna()
    .astype(str)
    .tolist()
)

stability = freq_df[
    freq_df["miRNA"]
    .astype(str)
    .isin(final_mirnas)
].copy()

stability["selection_count"] = pd.to_numeric(
    stability["selection_count"],
    errors="coerce"
)

stability = (
    stability
    .dropna(subset=["selection_count"])
    .sort_values(
        ["selection_count", "miRNA"],
        ascending=[True, True]
    )
)


stability_colors = [
    TEAL if x == 5 else GOLD
    for x in stability["selection_count"]
]


fig, ax = plt.subplots(figsize=(10, 9.5))

bars = ax.barh(
    stability["miRNA"],
    stability["selection_count"],
    color=stability_colors,
    edgecolor="none"
)

ax.axvline(
    5,
    linestyle="--",
    linewidth=1.2,
    color=GREY,
    alpha=0.7
)

ax.set_xlim(0, 5.55)
ax.set_xticks([0, 1, 2, 3, 4, 5])

ax.set_xlabel(
    "Number of outer training folds in which the miRNA was selected"
)

ax.set_ylabel("")

ax.set_title(
    "Selection stability of the final 30-miRNA panel",
    pad=14
)

clean_axes(ax, grid_axis="x")

for bar, value in zip(
    bars,
    stability["selection_count"]
):

    ax.text(
        value + 0.07,
        bar.get_y() + bar.get_height()/2,
        f"{int(value)}/5",
        va="center",
        fontsize=8.5,
        fontweight="bold",
        color=NAVY
    )

ax.text(
    0.99,
    -0.06,
    "Teal indicates selection in all five outer folds.",
    transform=ax.transAxes,
    ha="right",
    fontsize=8.5,
    color=GREY
)

fig.tight_layout()

save_fig(
    fig,
    "Fig3_final_panel_selection_stability"
)


# =====================================================================
# 7. FIGURE 4
# EXPRESSION DIFFERENCE
#
# IMPORTANT:
# This is NOT a conventional log2 fold-change.
# It is mean log2(RPM+1) tumor - mean log2(RPM+1) normal.
# =====================================================================

expression = bio_df.copy()

expression["log2FC_Tumor_vs_Normal"] = pd.to_numeric(
    expression["log2FC_Tumor_vs_Normal"],
    errors="coerce"
)

expression = (
    expression
    .dropna(
        subset=["log2FC_Tumor_vs_Normal"]
    )
    .sort_values(
        "log2FC_Tumor_vs_Normal",
        ascending=True
    )
)

exp_colors = [
    RED if x > 0 else BLUE
    for x in expression["log2FC_Tumor_vs_Normal"]
]


fig, ax = plt.subplots(figsize=(10.5, 10.5))

bars = ax.barh(
    expression["miRNA"],
    expression["log2FC_Tumor_vs_Normal"],
    color=exp_colors,
    edgecolor="none"
)

ax.axvline(
    0,
    color=NAVY,
    linewidth=1.0
)

ax.set_xlabel(
    "Mean expression difference: tumor − normal [log2(RPM+1)]"
)

ax.set_ylabel("")

ax.set_title(
    "Expression differences of the final 30-miRNA candidate panel",
    pad=14
)

clean_axes(ax, grid_axis="x")


max_abs = max(
    abs(expression["log2FC_Tumor_vs_Normal"].min()),
    abs(expression["log2FC_Tumor_vs_Normal"].max())
)

pad = max_abs * 0.03


for bar, value in zip(
    bars,
    expression["log2FC_Tumor_vs_Normal"]
):

    if value >= 0:

        x = value + pad
        ha = "left"

    else:

        x = value - pad
        ha = "right"

    ax.text(
        x,
        bar.get_y() + bar.get_height()/2,
        f"{value:.2f}",
        va="center",
        ha=ha,
        fontsize=7.5,
        color=NAVY
    )


ax.text(
    0.02,
    -0.06,
    "Blue: lower mean expression in tumor   •   Red: higher mean expression in tumor",
    transform=ax.transAxes,
    ha="left",
    fontsize=8.5,
    color=GREY
)

fig.tight_layout()

save_fig(
    fig,
    "Fig4_final_panel_expression_difference"
)


# =====================================================================
# 8. FIGURE 5
# IMPROVED VENN DIAGRAM
# =====================================================================

anova_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_ANOVA_full"
)

mi_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_MI_full"
)

mrmr_df = pd.read_excel(
    FEATURE_FILE,
    sheet_name="UB_O_mRMR_full"
)

A = set(
    anova_df["miRNA"].dropna().astype(str)
)

B = set(
    mi_df["miRNA"].dropna().astype(str)
)

C = set(
    mrmr_df["miRNA"].dropna().astype(str)
)


# matplotlib-venn varsa profesyonel Venn kullan.
try:

    from matplotlib_venn import venn3

    fig, ax = plt.subplots(figsize=(8.5, 7.8))

    v = venn3(
        subsets=(A, B, C),
        set_labels=("ANOVA", "Mutual information", "mRMR"),
        ax=ax
    )

    patch_colors = {
        "100": PALE_TEAL,
        "010": PALE_GOLD,
        "001": LIGHT_BLUE,
        "110": ORANGE,
        "101": TEAL,
        "011": GOLD,
        "111": RED
    }

    for region, color in patch_colors.items():

        patch = v.get_patch_by_id(region)

        if patch is not None:
            patch.set_facecolor(color)
            patch.set_edgecolor(WHITE)
            patch.set_linewidth(1.8)
            patch.set_alpha(0.88)

    for region in [
        "100", "010", "001",
        "110", "101", "011", "111"
    ]:

        txt = v.get_label_by_id(region)

        if txt is not None:

            txt.set_fontsize(
                13 if region != "111" else 15
            )

            txt.set_fontweight("bold")

            txt.set_color(NAVY)

    if v.set_labels is not None:

        for txt in v.set_labels:

            if txt is not None:
                txt.set_fontsize(12)
                txt.set_fontweight("bold")
                txt.set_color(NAVY)

    common_all = len(A & B & C)

    ax.set_title(
        "Overlap among the top 30 miRNAs identified by feature-selection methods",
        fontsize=14,
        fontweight="bold",
        pad=20
    )

    ax.text(
        0.5,
        -0.07,
        f"{common_all} miRNAs were shared by all three feature-selection methods.",
        transform=ax.transAxes,
        ha="center",
        fontsize=9.5,
        color=GREY
    )

    fig.tight_layout()

    save_fig(
        fig,
        "Fig5_ANOVA_MI_mRMR_Venn"
    )

except ImportError:

    # -------------------------------------------------------------
    # FALLBACK VERSION
    # If matplotlib-venn is not installed.
    # -------------------------------------------------------------

    from matplotlib.patches import Circle

    only_a = len(A - B - C)
    only_b = len(B - A - C)
    only_c = len(C - A - B)

    ab = len((A & B) - C)
    ac = len((A & C) - B)
    bc = len((B & C) - A)

    abc = len(A & B & C)

    fig, ax = plt.subplots(figsize=(8.5, 8))

    circles = [
        Circle(
            (0.38, 0.58),
            0.27,
            color=PALE_TEAL,
            alpha=0.65,
            ec=NAVY,
            lw=1.3
        ),
        Circle(
            (0.62, 0.58),
            0.27,
            color=PALE_GOLD,
            alpha=0.65,
            ec=NAVY,
            lw=1.3
        ),
        Circle(
            (0.50, 0.38),
            0.27,
            color=LIGHT_BLUE,
            alpha=0.65,
            ec=NAVY,
            lw=1.3
        )
    ]

    for circle in circles:
        ax.add_patch(circle)

    # Method labels
    ax.text(
        0.20, 0.84,
        "ANOVA",
        fontsize=12,
        fontweight="bold",
        color=NAVY
    )

    ax.text(
        0.68, 0.84,
        "Mutual\ninformation",
        fontsize=12,
        fontweight="bold",
        color=NAVY,
        ha="center"
    )

    ax.text(
        0.50, 0.06,
        "mRMR",
        fontsize=12,
        fontweight="bold",
        color=NAVY,
        ha="center"
    )

    positions = {
        "A": (0.23, 0.61, only_a),
        "B": (0.77, 0.61, only_b),
        "C": (0.50, 0.20, only_c),
        "AB": (0.50, 0.70, ab),
        "AC": (0.36, 0.41, ac),
        "BC": (0.64, 0.41, bc),
        "ABC": (0.50, 0.52, abc)
    }

    for key, (x, y, value) in positions.items():

        ax.text(
            x,
            y,
            str(value),
            fontsize=15 if key == "ABC" else 12,
            fontweight="bold",
            ha="center",
            va="center",
            color=NAVY
        )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    ax.set_title(
        "Overlap among the top 30 miRNAs identified by feature-selection methods",
        pad=20
    )

    ax.text(
        0.5,
        -0.02,
        f"{abc} miRNAs were shared by all three feature-selection methods.",
        transform=ax.transAxes,
        ha="center",
        fontsize=9.5,
        color=GREY
    )

    fig.tight_layout()

    save_fig(
        fig,
        "Fig5_ANOVA_MI_mRMR_Venn"
    )

    print(
        "\nNOT: Daha düzgün ölçekli Venn için istersen "
        "'pip install matplotlib-venn' kurabilirsin."
    )


common_all = len(A & B & C)

print(
    "\nANOVA ∩ MI ∩ mRMR:",
    common_all
)

print(
    sorted(A & B & C)
)


# =====================================================================
# 9. FIGURE 6
# FINAL OPTIMIZED MODEL PERFORMANCE
# =====================================================================

row = summary_df.iloc[0]

metrics = [
    "Accuracy",
    "F1-Score",
    "MCC",
    "Precision",
    "Recall",
    "ROC AUC"
]

metric_labels = [
    "Accuracy",
    "F1-score",
    "MCC",
    "Precision",
    "Recall",
    "ROC-AUC"
]

metric_values = [
    float(row[m])
    for m in metrics
]

metric_colors = [
    BLUE,
    TEAL,
    ORANGE,
    LIGHT_BLUE,
    GOLD,
    NAVY
]


fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    metric_labels,
    metric_values,
    color=metric_colors,
    edgecolor="none",
    width=0.65
)

ax.set_ylim(0.90, 1.01)

ax.set_ylabel("Cross-validated score")

ax.set_title(
    "Performance of the highest-ranking optimized model",
    pad=14
)

clean_axes(ax, grid_axis="y")


for bar, value in zip(
    bars,
    metric_values
):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.002,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=NAVY
    )


ax.text(
    0.5,
    -0.12,
    "Unbalanced data + mRMR + optimized AdaBoost",
    transform=ax.transAxes,
    ha="center",
    fontsize=9,
    color=GREY
)

fig.tight_layout()

save_fig(
    fig,
    "Fig6_best_optimized_model_performance"
)


# =====================================================================
# 10. PREPARE RAW DATA FOR OOF FINAL-MODEL ANALYSIS
#
# Exact preprocessing used in original analysis:
# raw counts -> RPM -> log2(RPM+1)
# keep RPM > 1 in >=20% samples
# =====================================================================

print("\n" + "=" * 75)
print("FINAL MODEL OOF ANALYSIS")
print("=" * 75)

df = pd.read_csv(CSV_FILE)

if "Sample" not in df.columns:
    raise KeyError(
        "'Sample' column not found."
    )

if "label" not in df.columns:
    raise KeyError(
        "'label' column not found."
    )


sample_ids = (
    df["Sample"]
    .astype(str)
    .reset_index(drop=True)
)

groups = (
    sample_ids
    .str[:12]
    .reset_index(drop=True)
)

y = (
    df["label"]
    .astype(int)
    .reset_index(drop=True)
)


feature_cols = [
    c for c in df.columns
    if c not in ["Sample", "label"]
]


counts = (
    df[feature_cols]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .fillna(0.0)
    .clip(lower=0)
)


library_sizes = counts.sum(axis=1)

library_sizes = library_sizes.replace(
    0,
    np.nan
)


rpm = (
    counts
    .div(
        library_sizes,
        axis=0
    )
    * 1e6
)

rpm = rpm.fillna(0.0)


log_rpm = np.log2(
    rpm + 1.0
)


minimum_fraction = 0.20

minimum_samples = int(
    np.ceil(
        len(df)
        * minimum_fraction
    )
)


keep_mask = (
    (rpm > 1)
    .sum(axis=0)
    >= minimum_samples
)


kept_features = (
    keep_mask[
        keep_mask
    ]
    .index
    .tolist()
)


X = (
    log_rpm[
        kept_features
    ]
    .reset_index(drop=True)
)


print("Samples             :", len(X))
print("Raw miRNA features  :", len(feature_cols))
print("Filtered features   :", len(kept_features))
print("Unique patients     :", groups.nunique())
print("Tumor samples       :", int((y == 1).sum()))
print("Normal samples      :", int((y == 0).sum()))


# =====================================================================
# 11. mRMR SELECTOR
# Same behavior as original analysis
# =====================================================================

class MRMRSelector(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        K=30
    ):
        self.K = K
        self.selected_features_ = None
        self.feature_names_in_ = None


    def fit(
        self,
        X,
        y
    ):

        if isinstance(
            X,
            pd.DataFrame
        ):

            X_df = (
                X.copy()
                .reset_index(drop=True)
            )

            self.feature_names_in_ = list(
                X_df.columns
            )

        else:

            X_df = pd.DataFrame(
                X,
                columns=[
                    f"f{i}"
                    for i
                    in range(
                        X.shape[1]
                    )
                ]
            )

            self.feature_names_in_ = list(
                X_df.columns
            )


        y_series = (
            pd.Series(y)
            .reset_index(drop=True)
        )


        self.selected_features_ = mrmr_classif(
            X=X_df,
            y=y_series,
            K=self.K
        )

        return self


    def transform(
        self,
        X
    ):

        if isinstance(
            X,
            pd.DataFrame
        ):

            X_df = X.copy()

        else:

            X_df = pd.DataFrame(
                X,
                columns=self.feature_names_in_
            )

        return X_df[
            self.selected_features_
        ]


# =====================================================================
# 12. EXACT FINAL MODEL
# =====================================================================

outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "feature_selection",
        MRMRSelector(K=30)
    ),
    (
        "classifier",
        AdaBoostClassifier(
            n_estimators=100,
            random_state=42
        )
    )
])


# EXACT grid from original code
param_grid = {
    "classifier__n_estimators":
        [50, 100, 200],

    "classifier__learning_rate":
        [0.01, 0.1, 1.0]
}


# =====================================================================
# 13. OOF STORAGE
# =====================================================================

n = len(X)

oof_pred = np.full(
    n,
    np.nan
)

oof_score = np.full(
    n,
    np.nan
)

oof_fold = np.full(
    n,
    -1,
    dtype=int
)

fold_results = []
best_params_rows = []


# =====================================================================
# 14. ONLY FINAL MODEL IS RE-RUN
# =====================================================================

for fold_id, (
    train_idx,
    test_idx
) in enumerate(
    outer_cv.split(
        X,
        y,
        groups
    ),
    start=1
):

    print(
        f"\nOuter fold {fold_id}/5"
    )

    X_train = (
        X.iloc[
            train_idx
        ]
        .reset_index(drop=True)
    )

    X_test = (
        X.iloc[
            test_idx
        ]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[
            train_idx
        ]
        .reset_index(drop=True)
    )

    y_test = (
        y.iloc[
            test_idx
        ]
        .reset_index(drop=True)
    )

    groups_train = (
        groups.iloc[
            train_idx
        ]
        .reset_index(drop=True)
    )

    groups_test = (
        groups.iloc[
            test_idx
        ]
        .reset_index(drop=True)
    )


    # Patient leakage check
    overlap = set(
        groups_train
    ).intersection(
        set(groups_test)
    )

    if overlap:

        raise RuntimeError(
            f"Patient leakage detected "
            f"in fold {fold_id}: "
            f"{len(overlap)} patients."
        )


    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="f1",
        n_jobs=1,
        error_score=0.0,
        refit=True
    )


    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )


    fitted_model = (
        search.best_estimator_
    )


    pred = fitted_model.predict(
        X_test
    )


    if hasattr(
        fitted_model,
        "predict_proba"
    ):

        score = (
            fitted_model
            .predict_proba(
                X_test
            )[:, 1]
        )

    elif hasattr(
        fitted_model,
        "decision_function"
    ):

        score = (
            fitted_model
            .decision_function(
                X_test
            )
        )

    else:

        score = pred.astype(float)


    # Save by original row index
    oof_pred[test_idx] = pred
    oof_score[test_idx] = score
    oof_fold[test_idx] = fold_id


    fold_auc = roc_auc_score(
        y_test,
        score
    )


    fold_result = {
        "Fold": fold_id,
        "Accuracy":
            accuracy_score(
                y_test,
                pred
            ),
        "F1-Score":
            f1_score(
                y_test,
                pred,
                zero_division=0
            ),
        "MCC":
            matthews_corrcoef(
                y_test,
                pred
            ),
        "Precision":
            precision_score(
                y_test,
                pred,
                zero_division=0
            ),
        "Recall":
            recall_score(
                y_test,
                pred,
                zero_division=0
            ),
        "ROC AUC":
            fold_auc
    }

    fold_results.append(
        fold_result
    )


    best_params_rows.append({
        "Fold": fold_id,
        **search.best_params_
    })


    print(
        "Best parameters:",
        search.best_params_
    )


# =====================================================================
# 15. CHECK OOF COMPLETENESS
# =====================================================================

if (
    np.isnan(oof_pred).any()
    or np.isnan(oof_score).any()
    or (oof_fold == -1).any()
):

    raise RuntimeError(
        "Some samples did not receive OOF predictions."
    )


oof_pred = oof_pred.astype(int)


# =====================================================================
# 16. OOF METRICS
# =====================================================================

fold_results_df = pd.DataFrame(
    fold_results
)

mean_fold_metrics = (
    fold_results_df[
        [
            "Accuracy",
            "F1-Score",
            "MCC",
            "Precision",
            "Recall",
            "ROC AUC"
        ]
    ]
    .mean()
)


pooled_metrics = {
    "Accuracy":
        accuracy_score(
            y,
            oof_pred
        ),

    "F1-Score":
        f1_score(
            y,
            oof_pred,
            zero_division=0
        ),

    "MCC":
        matthews_corrcoef(
            y,
            oof_pred
        ),

    "Precision":
        precision_score(
            y,
            oof_pred,
            zero_division=0
        ),

    "Recall":
        recall_score(
            y,
            oof_pred,
            zero_division=0
        ),

    "ROC AUC":
        roc_auc_score(
            y,
            oof_score
        )
}


print("\n" + "=" * 75)
print("MEAN OUTER-FOLD METRICS")
print("=" * 75)

print(
    mean_fold_metrics
    .round(4)
)


print("\n" + "=" * 75)
print("POOLED OOF METRICS")
print("=" * 75)

for key, value in pooled_metrics.items():

    print(
        f"{key:10s}: {value:.4f}"
    )


# =====================================================================
# 17. SAVE OOF DATA
# =====================================================================

oof_output = pd.DataFrame({
    "Sample":
        sample_ids,

    "Patient":
        groups,

    "Outer_Fold":
        oof_fold,

    "y_true":
        y,

    "y_pred":
        oof_pred,

    "y_score":
        oof_score
})


oof_output.to_csv(
    os.path.join(
        OUT_DIR,
        "OOF_predictions_final_mRMR_AdaBoost.csv"
    ),
    index=False
)


fold_results_df.to_csv(
    os.path.join(
        OUT_DIR,
        "OOF_fold_metrics_final_mRMR_AdaBoost.csv"
    ),
    index=False
)


pd.DataFrame(
    best_params_rows
).to_csv(
    os.path.join(
        OUT_DIR,
        "OOF_best_parameters_by_fold.csv"
    ),
    index=False
)


# =====================================================================
# 18. FIGURE 7
# BEAUTIFUL OOF CONFUSION MATRIX
# Count + within-class percentage in each cell
# =====================================================================

cm = confusion_matrix(
    y,
    oof_pred,
    labels=[0, 1]
)

row_totals = (
    cm.sum(
        axis=1,
        keepdims=True
    )
)

cm_pct = (
    cm
    / row_totals
    * 100
)


fig, ax = plt.subplots(
    figsize=(6.6, 5.8)
)

im = ax.imshow(
    cm,
    cmap="Blues"
)


ax.set_xticks(
    [0, 1],
    labels=[
        "Normal",
        "Tumor"
    ]
)

ax.set_yticks(
    [0, 1],
    labels=[
        "Normal",
        "Tumor"
    ]
)


ax.set_xlabel(
    "Predicted class",
    fontweight="bold"
)

ax.set_ylabel(
    "True class",
    fontweight="bold"
)

ax.set_title(
    "Out-of-fold confusion matrix",
    fontsize=14,
    fontweight="bold",
    pad=14
)


threshold = (
    cm.max()
    / 2
)


for i in range(2):

    for j in range(2):

        text_color = (
            WHITE
            if cm[i, j] > threshold
            else NAVY
        )

        ax.text(
            j,
            i,
            f"{cm[i, j]}\n"
            f"({cm_pct[i, j]:.1f}%)",
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold",
            color=text_color
        )


for spine in ax.spines.values():
    spine.set_visible(False)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.ax.set_ylabel(
    "Number of samples",
    rotation=270,
    labelpad=16
)


ax.text(
    0.5,
    -0.16,
    "Predictions were generated only when each sample belonged to the held-out outer fold.",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.5,
    color=GREY
)


fig.tight_layout()

save_fig(
    fig,
    "Fig7_OOF_confusion_matrix"
)


# =====================================================================
# 19. FIGURE 8
# OOF ROC CURVE
# =====================================================================

fpr, tpr, thresholds = roc_curve(
    y,
    oof_score
)

pooled_auc = roc_auc_score(
    y,
    oof_score
)


fig, ax = plt.subplots(
    figsize=(6.5, 6)
)


ax.plot(
    fpr,
    tpr,
    linewidth=2.5,
    color=TEAL,
    label=f"OOF ROC-AUC = {pooled_auc:.3f}"
)


ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.2,
    color=GREY,
    alpha=0.75,
    label="Chance level"
)


ax.set_xlim(
    -0.01,
    1.01
)

ax.set_ylim(
    -0.01,
    1.01
)


ax.set_xlabel(
    "False-positive rate"
)

ax.set_ylabel(
    "True-positive rate"
)

ax.set_title(
    "Out-of-fold ROC curve",
    pad=14
)

clean_axes(
    ax,
    grid_axis="both"
)

# clean_axes doesn't implement both, add manually
ax.grid(
    linestyle="--",
    linewidth=0.6,
    alpha=0.40,
    color=GRID
)

ax.set_axisbelow(True)


legend = ax.legend(
    loc="lower right",
    frameon=True
)

legend.get_frame().set_edgecolor(
    LIGHT_GREY
)

legend.get_frame().set_linewidth(
    0.8
)


ax.text(
    0.5,
    -0.14,
    "Unbalanced data + mRMR + optimized AdaBoost; patient-grouped nested cross-validation.",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.5,
    color=GREY
)


fig.tight_layout()

save_fig(
    fig,
    "Fig8_OOF_ROC_curve"
)


# =====================================================================
# 20. SAVE SUMMARY TABLE
# =====================================================================

summary_export = pd.DataFrame({
    "Metric":
        list(
            pooled_metrics.keys()
        ),

    "Pooled_OOF":
        list(
            pooled_metrics.values()
        ),

    "Mean_Outer_Fold":
        [
            float(
                mean_fold_metrics[m]
            )
            for m
            in pooled_metrics.keys()
        ]
})


summary_export.to_csv(
    os.path.join(
        OUT_DIR,
        "final_model_OOF_metric_summary.csv"
    ),
    index=False
)


# =====================================================================
# 21. FINAL CHECK
# =====================================================================

print("\n" + "=" * 75)
print("FINAL FIGURES SUCCESSFULLY CREATED")
print("=" * 75)

print("\nOutput directory:")
print(OUT_DIR)

print("\nCreated figures:")
print("  Fig1_TOPSIS_top10")
print("  Fig2_class_handling_TOPSIS")
print("  Fig3_final_panel_selection_stability")
print("  Fig4_final_panel_expression_difference")
print("  Fig5_ANOVA_MI_mRMR_Venn")
print("  Fig6_best_optimized_model_performance")
print("  Fig7_OOF_confusion_matrix")
print("  Fig8_OOF_ROC_curve")

print("\nAdditional reproducibility files:")
print("  OOF_predictions_final_mRMR_AdaBoost.csv")
print("  OOF_fold_metrics_final_mRMR_AdaBoost.csv")
print("  OOF_best_parameters_by_fold.csv")
print("  final_model_OOF_metric_summary.csv")

print("\nKey checks:")
print("Final panel size            :", len(final_mirnas))
print("ANOVA ∩ MI ∩ mRMR          :", common_all)
print("OOF samples predicted       :", len(oof_pred))
print("Unique patients             :", groups.nunique())
print("Pooled OOF ROC-AUC          :", f"{pooled_auc:.4f}")

print("\n" + "=" * 75)

FILES
CSV     : C:\Users\DURU VURAL\Desktop\proje\THCA_miRNA_unique_patients_only.csv
TOPSIS  : C:\Windows\System32\000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\topsis_sonuclari_sirali_30.xlsx
FEATURE : C:\Windows\System32\000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\selected_features_by_scenario_30.xlsx
OUTPUT  : C:\Users\DURU VURAL\Desktop\proje\FINAL_PUBLICATION_FIGURES

SAVED FINAL MODEL CHECK
Classifier    : AdaBoost
Data handling : Unbalanced
Selector      : mRMR

✓ Doğru eski/canonical sonuç seti bulundu.
✓ Fig1_TOPSIS_top10
✓ Fig2_class_handling_TOPSIS
✓ Fig3_final_panel_selection_stability
✓ Fig4_final_panel_expression_difference
✓ Fig5_ANOVA_MI_mRMR_Venn

ANOVA ∩ MI ∩ mRMR: 22
['hsa-mir-1179', 'hsa-mir-1258', 'hsa-mir-144', 'hsa-mir-146b', 'hsa-mir-181a-2', 'hsa-mir-181b-2', 'hsa-mir-221', 'hsa-mir-222', 'hsa-mir-26a-2', 'hsa-mir-28', 'hs

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.02it/s]


Best parameters: {'classifier__learning_rate': 1.0, 'classifier__n_estimators': 50}

Outer fold 2/5


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  9.50it/s]


Best parameters: {'classifier__learning_rate': 0.1, 'classifier__n_estimators': 200}

Outer fold 3/5


...

Best parameters: {'classifier__learning_rate': 1.0, 'classifier__n_estimators': 200}

Outer fold 4/5


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  8.90it/s]


Best parameters: {'classifier__learning_rate': 1.0, 'classifier__n_estimators': 100}

Outer fold 5/5


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.40it/s]


Best parameters: {'classifier__learning_rate': 1.0, 'classifier__n_estimators': 100}

MEAN OUTER-FOLD METRICS
Accuracy     0.9912
F1-Score     0.9951
MCC          0.9536
Precision    0.9961
Recall       0.9941
ROC AUC      0.9968
dtype: float64

POOLED OOF METRICS
Accuracy  : 0.9912
F1-Score  : 0.9951
MCC       : 0.9531
Precision : 0.9960
Recall    : 0.9941
ROC AUC   : 0.9930
✓ Fig7_OOF_confusion_matrix
✓ Fig8_OOF_ROC_curve

FINAL FIGURES SUCCESSFULLY CREATED

Output directory:
C:\Users\DURU VURAL\Desktop\proje\FINAL_PUBLICATION_FIGURES

Created figures:
  Fig1_TOPSIS_top10
  Fig2_class_handling_TOPSIS
  Fig3_final_panel_selection_stability
  Fig4_final_panel_expression_difference
  Fig5_ANOVA_MI_mRMR_Venn
  Fig6_best_optimized_model_performance
  Fig7_OOF_confusion_matrix
  Fig8_OOF_ROC_curve

Additional reproducibility files:
  OOF_predictions_final_mRMR_AdaBoost.csv
  OOF_fold_metrics_final_mRMR_AdaBoost.csv
  OOF_best_parameters_by_fold.csv
  final_model_OOF_metric_summary.csv

Key